In [1]:
import numpy as np
import gymnasium as gym
import random
import pandas as pd

env = gym.make("FrozenLake-v1")

alpha = 0.85
gamma = 0.90
epsilon = 0.8

num_episodes = 5000
num_timesteps = 1000

def random_policy():
    return env.action_space.sample()

v = {s: 0.0 for s in range(env.observation_space.n)}

for i in range(num_episodes):
    s, info = env.reset()

    for t in range(num_timesteps):
        a = random_policy()
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        v[s] += alpha * (r + gamma * v[s_] - v[s])
        s = s_

        if done:
            break

print("TD Prediction:")
print(pd.DataFrame(list(v.items()), columns=["State", "Value"]))

Q = {
    (s, a): 0.0
    for s in range(env.observation_space.n)
    for a in range(env.action_space.n)
}

def epsilon_greedy(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    return max(
        range(env.action_space.n),
        key=lambda a: Q[(state, a)]
    )

for i in range(num_episodes):
    s, info = env.reset()
    a = epsilon_greedy(s, epsilon)

    for t in range(num_timesteps):
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        a_ = epsilon_greedy(s_, epsilon)

        target = r if done else r + gamma * Q[(s_, a_)]
        Q[(s, a)] += alpha * (target - Q[(s, a)])

        s = s_
        a = a_

        if done:
            break

print("\nSARSA:")
print(pd.DataFrame(
    [(s, a, value) for (s, a), value in Q.items()],
    columns=["State", "Action", "Q-Value"]
))

for i in range(num_episodes):
    s, info = env.reset()

    for t in range(num_timesteps):
        a = epsilon_greedy(s, epsilon)
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        target = r if done else r + gamma * max(
            Q[(s_, next_a)] for next_a in range(env.action_space.n)
        )

        Q[(s, a)] += alpha * (target - Q[(s, a)])
        s = s_

        if done:
            break

print("\nQ-Learning:")
print(pd.DataFrame(
    [(s, a, value) for (s, a), value in Q.items()],
    columns=["State", "Action", "Q-Value"]
))

env.close()

TD Prediction:
    State         Value
0       0  1.229337e-05
1       1  2.085406e-06
2       2  1.780183e-06
3       3  6.254983e-07
4       4  1.360604e-06
5       5  0.000000e+00
6       6  3.294922e-07
7       7  0.000000e+00
8       8  1.482035e-02
9       9  9.749967e-03
10     10  8.075872e-01
11     11  0.000000e+00
12     12  0.000000e+00
13     13  1.286786e-02
14     14  2.013771e-01
15     15  0.000000e+00

SARSA:
    State  Action   Q-Value
0       0       0  0.000589
1       0       1  0.000163
2       0       2  0.000300
3       0       3  0.001000
4       1       0  0.000003
..    ...     ...       ...
59     14       3  0.016812
60     15       0  0.000000
61     15       1  0.000000
62     15       2  0.000000
63     15       3  0.000000

[64 rows x 3 columns]

Q-Learning:
    State  Action   Q-Value
0       0       0  0.309606
1       0       1  0.278247
2       0       2  0.258235
3       0       3  0.259090
4       1       0  0.039599
..    ...     ...       ...
5

In [3]:
import random
import gymnasium as gym

env = gym.make("Taxi-v4", render_mode="ansi")

q = {}

for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        q[(s, a)] = 0.0


def update_q_table(prev_state, action, reward, nextstate, alpha, gamma, done):
    if done:
        qa = 0
    else:
        qa = max(
            q[(nextstate, a)]
            for a in range(env.action_space.n)
        )

    q[(prev_state, action)] += alpha * (
        reward + gamma * qa - q[(prev_state, action)]
    )


def epsilon_greedy_policy(state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()

    return max(
        range(env.action_space.n),
        key=lambda a: q[(state, a)]
    )


alpha = 0.4
gamma = 0.999
epsilon = 0.017
num_episodes = 8000

for i in range(num_episodes):
    total_reward = 0
    prev_state, info = env.reset()

    while True:
        action = epsilon_greedy_policy(prev_state, epsilon)

        nextstate, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        update_q_table(
            prev_state,
            action,
            reward,
            nextstate,
            alpha,
            gamma,
            done
        )

        prev_state = nextstate
        total_reward += reward

        if done:
            break

    print("Episode:", i + 1, "Total reward:", total_reward)

print("\nTraining completed!")

state, info = env.reset()
print(env.render())

while True:
    action = max(
        range(env.action_space.n),
        key=lambda a: q[(state, a)]
    )

    nextstate, reward, terminated, truncated, info = env.step(action)
    state = nextstate

    print(env.render())

    if terminated or truncated:
        print("Final Reward:", reward)
        break

env.close()

Streaming output truncated to the last 5000 lines.
Episode: 3157 Total reward: 4
Episode: 3158 Total reward: 7
Episode: 3159 Total reward: 5
Episode: 3160 Total reward: 10
Episode: 3161 Total reward: 5
Episode: 3162 Total reward: 4
Episode: 3163 Total reward: 12
Episode: 3164 Total reward: 7
Episode: 3165 Total reward: 8
Episode: 3166 Total reward: -4
Episode: 3167 Total reward: 12
Episode: 3168 Total reward: 7
Episode: 3169 Total reward: 10
Episode: 3170 Total reward: 4
Episode: 3171 Total reward: 10
Episode: 3172 Total reward: 5
Episode: 3173 Total reward: 12
Episode: 3174 Total reward: 8
Episode: 3175 Total reward: 7
Episode: 3176 Total reward: 8
Episode: 3177 Total reward: 3
Episode: 3178 Total reward: 8
Episode: 3179 Total reward: 5
Episode: 3180 Total reward: 4
Episode: 3181 Total reward: 7
Episode: 3182 Total reward: 7
Episode: 3183 Total reward: 4
Episode: 3184 Total reward: 6
Episode: 3185 Total reward: -3
Episode: 3186 Total reward: 12
Episode: 3187 Total reward: 5
Episode: 3